# Dafne Thigh — Sheffield Evaluation — GPU Lambda

GPU-accelerated version of `column_compare_dafne_sheffield.ipynb` designed to run
on a Lambda instance alongside the inference notebooks.

**Speedup over CPU original:**
- `boundary_iou_3d`: pure-Python nested loops → 3-D `max_pool` kernel (orders of magnitude faster)
- `extract_boundary_3d`: pure-Python loops → GPU erosion via inverted max-pool
- Hausdorff: surface points extracted on GPU, pairwise distances via `torch.cdist` with chunking
- Dice, BCE, Jaccard, vol-similarity, FP/FN, inter-slice Dice: plain tensor ops on GPU

## Upload to Lambda
```bash
# GT DICOMs
rsync -avz -e "ssh -i /tmp/lambda_key -o StrictHostKeyChecking=no" \
  /tmp/docker-desktop-root/run/desktop/mnt/host/c/Projects/dissector/eval_notebooks/sheffeld/20440203/ \
  ubuntu@<IP>:~/sheffeld/20440203/

# Dafne predictions (if not already there from inference)
rsync -avz -e "ssh -i /tmp/lambda_key -o StrictHostKeyChecking=no" \
  /tmp/docker-desktop-root/run/desktop/mnt/host/c/Projects/dissector/eval_notebooks/dafne/sheffield_segs/ \
  ubuntu@<IP>:~/dafne_sheffield_segs/
```

## Download results
```bash
rsync -avz --mkpath -e "ssh -i /tmp/lambda_key -o StrictHostKeyChecking=no" \
  ubuntu@<IP>:~/dafne_sheffield_results/ \
  /tmp/docker-desktop-root/run/desktop/mnt/host/c/Projects/dissector/eval_notebooks/dafne/codes/results_sheffield/
```

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                       'pydicom', 'SimpleITK', 'pandas', 'numpy<2'])
print('Dependencies ready.')

In [ ]:
import glob, os, re
import numpy as np
import pandas as pd
import pydicom
import torch
import torch.nn.functional as F

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'  {torch.cuda.get_device_name(0)}')
    free, total = torch.cuda.mem_get_info(0)
    print(f'  VRAM: {free/1e9:.1f} GB free / {total/1e9:.1f} GB total')

In [ ]:
# ── Configuration ──────────────────────────────────────────────────────────────
BOUNDARY_DISTANCE = 1

GT_DIR     = os.path.expanduser('~/sheffeld/20440203')
SEG_DIR    = os.path.expanduser('~/dafne_sheffield_segs')
RESULT_DIR = os.path.expanduser('~/dafne_sheffield_results')
ALGO_TAG   = 'dafne'
os.makedirs(RESULT_DIR, exist_ok=True)

# (muscle_name, sheffield_gt_label, npz_keys_to_OR)
# Sheffield GT is bilateral — OR both laterality keys before comparing.
# NPZ keys use spaces, e.g. 'Adductor Longus_L' (confirmed from sample files).
MUSCLES = [
    ('adductor_longus',      2,  ['Adductor Longus_L',      'Adductor Longus_R'     ]),
    ('adductor_magnus',      3,  ['Adductor Magnus_L',      'Adductor Magnus_R'     ]),
    ('biceps_femoris_short', 4,  ['Biceps Femoris Short_L', 'Biceps Femoris Short_R']),
    ('biceps_femoris_long',  5,  ['Biceps Femoris Long_L',  'Biceps Femoris Long_R' ]),
    ('gracilis',             16, ['Gracilis_L',             'Gracilis_R'            ]),
    ('rectus_femoris',       27, ['Rectus Femoris_L',       'Rectus Femoris_R'      ]),
    ('sartorius',            28, ['Sartorius_L',            'Sartorius_R'           ]),
    ('semimembranosus',      29, ['Semimembranosus_L',      'Semimembranosus_R'     ]),
    ('semitendinosus',       30, ['Semitendinosus_L',       'Semitendinosus_R'      ]),
    ('vastus_intermedius',   35, ['Vastus Intermedius_L',   'Vastus Intermedius_R'  ]),
    ('vastus_lateralis',     36, ['Vastus Lateralis_L',     'Vastus Lateralis_R'    ]),
    ('vastus_medialis',      37, ['Vastus Medialis_L',      'Vastus Medialis_R'     ]),
]

seg_files = sorted(glob.glob(os.path.join(SEG_DIR, 'Aug_*', 'Aug_*_dafne_thigh.npz')))
print(f'GT dir : {GT_DIR}')
print(f'Seg dir: {SEG_DIR}')
print(f'Found  : {len(seg_files)} NPZ files')
if seg_files:
    s = np.load(seg_files[0])
    print(f'Sample keys: {sorted(s.files)}')

In [ ]:
# ── GPU metric helpers ─────────────────────────────────────────────────────────

def _to_bool(arr, device):
    """numpy uint8 → bool CUDA tensor (D,H,W)."""
    return torch.from_numpy(arr.astype(np.uint8)).bool().to(device)


def _erode3d(mask):
    """3-D morphological erosion via max-pool on inverted mask."""
    inv = (~mask).float().view(1, 1, *mask.shape)
    inv_pad = F.pad(inv, [1, 1, 1, 1, 1, 1], value=0)
    eroded_inv = F.max_pool3d(inv_pad, kernel_size=3, stride=1, padding=0)
    return ~(eroded_inv.squeeze(0).squeeze(0) > 0)


def _dilate3d(mask, dist=1):
    """3-D morphological dilation via iterated max-pool."""
    m = mask.float().view(1, 1, *mask.shape)
    for _ in range(dist):
        m_pad = F.pad(m, [1, 1, 1, 1, 1, 1], value=0)
        m = F.max_pool3d(m_pad, kernel_size=3, stride=1, padding=0)
    return m.squeeze(0).squeeze(0) > 0


def _surface3d(mask):
    """Surface voxels = mask AND NOT eroded(mask)."""
    return mask & ~_erode3d(mask)


def _surface_pts(mask, spacing):
    """Coordinates of surface voxels scaled by voxel spacing.
    spacing: (col_mm, row_mm, slice_mm) — SimpleITK convention.
    numpy array is (D, H, W) = (slice, row, col).
    """
    surf = _surface3d(mask)
    if surf.sum() == 0:
        return None
    idx  = surf.nonzero(as_tuple=False).float()   # (N, 3): [d, h, w]
    # scale: d→slice_mm, h→row_mm, w→col_mm
    scale = torch.tensor(
        [spacing[2], spacing[1], spacing[0]],
        dtype=torch.float32, device=mask.device
    )
    return idx * scale


def hausdorff_gpu(pred, gt, spacing, chunk=3000):
    """95th-percentile Hausdorff distance on GPU via chunked torch.cdist."""
    if pred.sum() == 0 or gt.sum() == 0:
        return float('nan')
    pp = _surface_pts(pred, spacing)
    gp = _surface_pts(gt,   spacing)
    if pp is None or gp is None:
        return float('nan')

    def directed_max(a, b):
        min_dists = []
        for i in range(0, len(a), chunk):
            d = torch.cdist(a[i:i+chunk], b)   # (chunk, |b|)
            min_dists.append(d.min(dim=1).values)
        return torch.cat(min_dists).max().item()

    return max(directed_max(pp, gp), directed_max(gp, pp))


def dice_gpu(pred, gt):
    inter = (pred & gt).float().sum()
    denom = pred.float().sum() + gt.float().sum()
    return (2 * inter / (denom + 1e-8)).item()


def jaccard_gpu(pred, gt):
    inter = (pred & gt).float().sum()
    union = (pred | gt).float().sum()
    return (inter / (union + 1e-8)).item()


def volume_similarity_gpu(pred, gt):
    ps = pred.float().sum()
    gs = gt.float().sum()
    return (1 - (ps - gs).abs() / (ps + gs + 1e-8)).item()


def false_negative_gpu(pred, gt):
    fn = (~pred & gt).float().sum()
    return (fn / (gt.float().sum() + 1e-8)).item()


def false_positive_gpu(pred, gt):
    fp = (pred & ~gt).float().sum()
    return (fp / (pred.float().sum() + 1e-8)).item()


def bce_gpu(pred, gt, eps=1e-7):
    pf = pred.float().clamp(eps, 1 - eps)
    gf = gt.float()
    return (-gf * pf.log() - (1 - gf) * (1 - pf).log()).mean().item()


def boundary_iou_3d_gpu(pred, gt, dist=1):
    """Boundary IoU: dilated surface of pred vs dilated surface of gt."""
    b_pred = _dilate3d(_surface3d(pred), dist)
    b_gt   = _dilate3d(_surface3d(gt),   dist)
    inter  = (b_pred & b_gt).float().sum()
    union  = (b_pred | b_gt).float().sum()
    return (inter / (union + 1e-8)).item()


def inter_slice_dice_gpu(pred):
    """Mean Dice between adjacent slices; measures segmentation smoothness."""
    if pred.shape[0] < 2:
        return float('nan')
    a = pred[:-1].float()
    b = pred[1:].float()
    inter  = (a * b).sum(dim=(1, 2))
    denom  = a.sum(dim=(1, 2)) + b.sum(dim=(1, 2))
    valid  = denom > 0
    if not valid.any():
        return 0.0
    scores = (2 * inter[valid] / denom[valid])
    return scores.mean().item()


print('GPU metric helpers defined.')

In [ ]:
# ── DICOM I/O helpers ──────────────────────────────────────────────────────────

def read_gt(idx):
    ds  = pydicom.dcmread(os.path.join(GT_DIR, f'Aug_{idx}_segmentations.dcm'))
    raw = ds.pixel_array.astype(np.float32)
    if raw.ndim == 2:
        raw = raw[np.newaxis]
    labeled = np.round(raw * 37.0 / 255.0).astype(np.int32)
    labeled[raw == 0] = 0
    return np.clip(labeled, 0, 37)


def get_spacing(idx):
    """Return (col_mm, row_mm, slice_mm) — SimpleITK / ITK convention."""
    ds = pydicom.dcmread(os.path.join(GT_DIR, f'Aug_{idx}_segmentations.dcm'))
    ps = getattr(ds, 'PixelSpacing', [1.0, 1.0])
    st = float(getattr(ds, 'SliceThickness', 1.0))
    return (float(ps[1]), float(ps[0]), st)   # (col, row, slice)


print('DICOM helpers defined.')

In [ ]:
# ── Main evaluation loop ───────────────────────────────────────────────────────

def evaluate_muscle(muscle_name, sheffield_label, npz_keys):
    results = []

    for seg_path in seg_files:
        m = re.search(r'Aug_(\d+)', seg_path.replace('\\', '/'))
        if not m:
            continue
        idx = m.group(1)
        gt_path = os.path.join(GT_DIR, f'Aug_{idx}_segmentations.dcm')
        if not os.path.exists(gt_path):
            print(f'  [skip] GT missing: Aug_{idx}')
            continue

        gt_arr  = read_gt(idx)
        spacing = get_spacing(idx)

        # Build binary prediction: OR all laterality keys
        npz_data = np.load(seg_path)
        pred_np  = np.zeros(gt_arr.shape, dtype=np.uint8)
        for key in npz_keys:
            if key in npz_data.files:
                pred_np |= npz_data[key].astype(np.uint8)
            else:
                print(f'  [Aug_{idx}] missing NPZ key "{key}"')

        gt_bin = (gt_arr == sheffield_label).astype(np.uint8)

        # Move to GPU
        pred_t = _to_bool(pred_np, DEVICE)
        gt_t   = _to_bool(gt_bin,  DEVICE)

        with torch.no_grad():
            row = {
                'sample':                          f'Aug_{idx}',
                f'{muscle_name}_dice':             dice_gpu(pred_t, gt_t),
                f'{muscle_name}_hausdorff':        hausdorff_gpu(pred_t, gt_t, spacing),
                f'{muscle_name}_jaccard':          jaccard_gpu(pred_t, gt_t),
                f'{muscle_name}_volume_similarity':volume_similarity_gpu(pred_t, gt_t),
                f'{muscle_name}_false_negative':   false_negative_gpu(pred_t, gt_t),
                f'{muscle_name}_false_positive':   false_positive_gpu(pred_t, gt_t),
                f'{muscle_name}_bce':              bce_gpu(pred_t, gt_t),
                f'{muscle_name}_boundary_iou_3d':  boundary_iou_3d_gpu(pred_t, gt_t, BOUNDARY_DISTANCE),
                f'{muscle_name}_inter_slice_dice_pred': inter_slice_dice_gpu(pred_t),
                f'{muscle_name}_inter_slice_dice_gt':   inter_slice_dice_gpu(gt_t),
            }

        results.append(row)
        dice_val = row[f'{muscle_name}_dice']
        hd_val   = row[f'{muscle_name}_hausdorff']
        print(f'  Aug_{idx:>3s}  dice={dice_val:.4f}  hd={hd_val:.2f}mm')

    df       = pd.DataFrame(results)
    csv_path = os.path.join(RESULT_DIR, f'df_{muscle_name}_{ALGO_TAG}_sheffield.csv')
    df.to_csv(csv_path, index=False)
    print(f'  Saved {len(df)} rows → {csv_path}')
    return df


dfs = {}
for muscle_name, sheffield_label, npz_keys in MUSCLES:
    print(f'\n── {muscle_name}  (GT label={sheffield_label}, keys={npz_keys}) ──')
    dfs[muscle_name] = evaluate_muscle(muscle_name, sheffield_label, npz_keys)

print('\nDone.')

In [ ]:
summary_rows = []
for muscle_name, df in dfs.items():
    if df.empty:
        continue
    summary_rows.append({
        'muscle':         muscle_name,
        'n':              len(df),
        'dice_mean':      df[f'{muscle_name}_dice'].mean(),
        'dice_std':       df[f'{muscle_name}_dice'].std(),
        'hausdorff_mean': df[f'{muscle_name}_hausdorff'].mean(),
        'hausdorff_std':  df[f'{muscle_name}_hausdorff'].std(),
    })

summary      = pd.DataFrame(summary_rows).set_index('muscle')
summary_path = os.path.join(RESULT_DIR, f'summary_{ALGO_TAG}_sheffield.csv')
summary.to_csv(summary_path)
print(f'Summary saved → {summary_path}\n')
print(summary.round(4).to_string())